# 04 — Conditional Stacking Ensemble

Trains a stacking ensemble (LR + SVC + NB base models, LR meta-learner) on WoT data.
A hard gate prevents proceeding past Section 1 unless stacking beats the best single model by ≥ 1 pp macro F1.

## Section 0: Imports & Load Best Baseline

In [ ]:
# Stacking ensemble — train base models (LR+SVC+NB), generate out-of-fold meta-features,
# train meta-learner (LR) on OOF predictions. Gate: only proceed if +1pp vs baseline.
import warnings, json
from pathlib import Path
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score, classification_report
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler
import sys
sys.path.insert(0, str(Path('..').resolve()))
from src.loaders import load_wot
from src.pipelines import build_pipe
from src.scoring import holdout_score, append_registry
from src.label_schemes import WOT_SCHEMES, WOT_CLASS_NAMES

CONFIG = {
    'seed': 7524,
    'cv_folds': 5,
    'text_col': 'clean_message',
    'label_col': 'label',
    'improvement_threshold': 0.01,
    'registry_path': Path('../data/results/results_registry.csv'),
    'models_dir': Path('../models'),
}
seed = CONFIG['seed']
cv   = StratifiedKFold(n_splits=CONFIG['cv_folds'], shuffle=True, random_state=seed)
np.random.seed(seed)

# Load baseline from registry — highest test_macro_f1 across ml_pipeline + error_analysis_fix
registry  = pd.read_csv(CONFIG['registry_path'])
best_row  = (registry[registry['experiment'].isin(['ml_pipeline_multiclass','error_analysis_fix'])
                      & registry['test_macro_f1'].notna()]
             .sort_values('test_macro_f1', ascending=False).iloc[0])
wot_n     = int(best_row['n_classes']) if best_row['train_game']=='WoT' else 2
baseline  = float(best_row['test_macro_f1'])
print(f'Baseline to beat: {best_row["model"]} test_macro_f1={baseline:.4f} (game={best_row["train_game"]}, n={wot_n})')
print(f'Must exceed: {baseline + CONFIG["improvement_threshold"]:.4f}')

Stacking approach — out-of-fold meta-features prevent meta-learner from seeing training data directly. `CalibratedClassifierCV` needed for SVC `predict_proba`.

## Section 1: Stacking — Out-of-Fold Meta-Features

In [ ]:
# Out-of-fold stacking: meta-learner trains on cross_val_predict OOF probabilities.
# CRITICAL: meta-learner NEVER sees in-sample predictions — prevents overfitting.
# CalibratedClassifierCV used only for SVC here (needs predict_proba for stacking).
wot_train = load_wot('train')
wot_train = wot_train.copy()
wot_train['label'] = wot_train['label'].astype(int).map(WOT_SCHEMES[wot_n])
wot_val   = load_wot('val')
wot_val   = wot_val.copy()
wot_val['label'] = wot_val['label'].astype(int).map(WOT_SCHEMES[wot_n])
X_train   = wot_train[CONFIG['text_col']]
y_train   = wot_train[CONFIG['label_col']]
X_val     = wot_val[CONFIG['text_col']]
y_val     = wot_val[CONFIG['label_col']]

base_pipes = {
    'LR':  build_pipe(LogisticRegression(C=1.0, max_iter=1000, random_state=seed, n_jobs=1),
                       oversampler=RandomOverSampler(random_state=seed)),
    'NB':  build_pipe(MultinomialNB(alpha=0.1),
                       oversampler=RandomOverSampler(random_state=seed)),
    'SVC': build_pipe(CalibratedClassifierCV(
                          LinearSVC(C=1.0, max_iter=2000, tol=1e-3, random_state=seed),
                          cv=3),
                       oversampler=RandomOverSampler(random_state=seed)),
}

print('Generating out-of-fold predictions (this may take a few minutes)...')
oof_preds = {}
for name, pipe in base_pipes.items():
    oof = cross_val_predict(pipe, X_train, y_train, cv=cv, method='predict_proba', n_jobs=1)
    oof_preds[name] = oof
    print(f'  {name}: OOF shape={oof.shape}')

meta_X_train = np.hstack([oof_preds[n] for n in base_pipes])
meta_clf     = LogisticRegression(C=1.0, max_iter=1000, random_state=seed)
meta_scores  = cross_val_score(meta_clf, meta_X_train, y_train, cv=cv,
                                scoring='f1_macro', n_jobs=1)
stacking_cv_f1 = meta_scores.mean()
delta = stacking_cv_f1 - baseline
print(f'\nStacking cv_macro_f1: {stacking_cv_f1:.4f} ± {meta_scores.std():.4f}')
print(f'Baseline:             {baseline:.4f}')
print(f'Delta:                {delta:+.4f}')
print(f'Gate threshold:       {CONFIG["improvement_threshold"]}')

OOF meta-features: each base model generates probability predictions via 5-fold cross-validation. These OOF probas are horizontally stacked to form the meta-learner's training set, ensuring no data leakage.

## Section 2: Gate Check and Evaluation

In [ ]:
# Gate: proceed only if stacking beats baseline by >= improvement_threshold.
# Both outcomes logged to registry for dashboard transparency.
delta = stacking_cv_f1 - baseline

if delta < CONFIG['improvement_threshold']:
    print(f'✗ Stacking improvement {delta:+.4f} < {CONFIG["improvement_threshold"]:.2f} threshold.')
    print('  Skipping full evaluation. Best single model from notebooks 02/06 remains recommended.')
    append_registry({
        'experiment':          'ensemble',
        'train_game':          'WoT', 'test_game': 'WoT',
        'n_classes':           wot_n, 'label_scheme': 'native',
        'model':               'Stacking(LR+SVC+NB)',
        'cv_macro_f1':         round(stacking_cv_f1, 4), 'cv_std': round(meta_scores.std(), 4),
        'cv_weighted_f1':      None, 'cv_recall_macro': None, 'cv_precision_macro': None,
        'test_macro_f1':       None, 'test_weighted_f1': None,
        'test_precision_macro': None, 'test_recall_macro': None, 'test_auc': None,
        'per_class_recall':    None,
        'ood_macro_f1':        None, 'ood_weighted_f1': None,
        'ood_precision_macro': None, 'ood_recall_macro': None, 'ood_auc': None,
        'anomaly_auroc':       None,
        'notes':               'no_improvement_below_1pp_threshold',
    }, path=CONFIG['registry_path'])
else:
    print(f'✓ Improvement {delta:+.4f} >= threshold. Running full holdout evaluation.')
    for name, pipe in base_pipes.items():
        pipe.fit(X_train, y_train)
    meta_X_val = np.hstack([pipe.predict_proba(X_val) for pipe in base_pipes.values()])
    meta_clf.fit(meta_X_train, y_train)
    y_pred_val = meta_clf.predict(meta_X_val)
    test_f1 = float(f1_score(y_val, y_pred_val, average='macro', zero_division=0))
    print(f'\nStacking test_macro_f1: {test_f1:.4f}')
    print(classification_report(y_val, y_pred_val,
                                  target_names=WOT_CLASS_NAMES[wot_n], zero_division=0))
    append_registry({
        'experiment':          'ensemble',
        'train_game':          'WoT', 'test_game': 'WoT',
        'n_classes':           wot_n, 'label_scheme': 'native',
        'model':               'Stacking(LR+SVC+NB)',
        'cv_macro_f1':         round(stacking_cv_f1, 4), 'cv_std': round(meta_scores.std(), 4),
        'cv_weighted_f1':      None, 'cv_recall_macro': None, 'cv_precision_macro': None,
        'test_macro_f1':       round(test_f1, 4), 'test_weighted_f1': None,
        'test_precision_macro': None, 'test_recall_macro': None, 'test_auc': None,
        'per_class_recall':    None,
        'ood_macro_f1':        None, 'ood_weighted_f1': None,
        'ood_precision_macro': None, 'ood_recall_macro': None, 'ood_auc': None,
        'anomaly_auroc':       None,
        'notes':               f'improvement_{delta:+.4f}',
    }, path=CONFIG['registry_path'])

Both outcomes documented in registry. If gate fails, best single model from notebooks 02/06 is the final recommendation.